In [2]:
from pymongo import MongoClient
import pandas as pd, json

In [3]:
# Connect to the MongoDB server (localhost by default)
mg_client1 = MongoClient('mongodb://localhost:27017/', username='root', password='rootpwd')

##provide feedback to the user
if mg_client1:
    print("connected to mongo database successully.")
else:
    print("failed to connect to mongo database.")

connected to mongo database successully.


In [4]:
db = mg_client1["socialmedia"]

In [ ]:
"""
QUERY 1:

most active users by total engagement, aka the combined total of likes, shares, and comments
"""

query1 = [
  {"$match": {}},
  {"$group": {
    "_id": "$UserID", # group posts by user id

    # get aggregates
    "TotalComments": {"$sum": {"$size": "$Comments"}},
    "TotalLikes": {"$sum": "$LikeCount"},
    "TotalShares": {"$sum": "$ShareCount"}
  }},
  {"$project": {
    "TotalEngagement": {
      "$add": ["$TotalComments", "$TotalLikes", "$TotalShares"] # get overall engagement total
    }
  }},
  {"$lookup": { # join posts with users
    "from": "users",
    "localField": "_id",
    "foreignField": "UserID",
    "as": "userInfo"
  }},
  {"$unwind": "$userInfo"}, # flatten array
  {"$project": { # select statement
    "_id": 0,
    "UserID": "$_id",
    "Username": "$userInfo.Username",
    "TotalEngagement": 1
  }},
  {"$sort": {"TotalEngagement": -1}} # sort by total engagement
]

result1 = pd.DataFrame(db.posts.aggregate(query1))
result1

,TotalEngagement,UserID,Username
0,644,1016,aarav_mehta43
1,616,1092,vihaan_gupta29
2,598,1067,ananya_verma92
3,589,1049,kiara_patel18
4,585,1037,aarav_reddy87
...,...,...,...
95,29,1062,aditya_patel55
96,21,1005,reyansh_reddy30
97,20,1013,ayaan_reddy14
98,18,1100,kiara_reddy40


In [ ]:
"""
QUERY 2:

trending hashtags by frequency
"""
from datetime import datetime, timedelta


# we made the data march 3rd, so we'll percieve trending as for the last 2 weeks for when the data was made.
cutoff = datetime(2026, 3, 3) - timedelta(days=14)

query2 = [
  {"$match": {"Timestamp": {"$gte": cutoff}}}, # cutoff
  {"$unwind": "$Hashtags"},
  {"$group": {
    "_id": "$Hashtags.HashtagID",
    "usageCount": {"$sum": 1}
  }},
  {"$sort": {"usageCount": -1}}
]

result2 = pd.DataFrame(db.posts.aggregate(query2))

result2


,_id,usageCount
0,HASH-105,13
1,HASH-102,11
2,HASH-112,11
3,HASH-106,11
4,HASH-108,10
5,HASH-113,10
6,HASH-103,9
7,HASH-101,7
8,HASH-109,7
9,HASH-111,7


In [25]:
"""
QUERY 3:

Average engagement values per post type
"""

query3 = [
  {"$group": {
    "_id": "$PostType", # group posts by post type
    "AvgLikes": {"$avg": "$LikeCount"},
    "AvgShares": {"$avg": "$ShareCount"},
    "AvgComments": {"$avg": {"$size": "$Comments"}}
  }},
  {"$sort": {"avgLikes": -1}}
]

result3 = pd.DataFrame(db.posts.aggregate(query3))
result3

,_id,AvgLikes,AvgShares,AvgComments
0,Video,9.935127,0.310127,1.849684
1,Story,9.775081,0.294498,1.794498
2,Text,10.343879,0.261348,1.620358
3,Image,9.606432,0.280245,1.569678


In [ ]:
"""
QUERY 4:

Top commenters using window functions
"""

query4 = [
  {"$unwind": "$Comments"}, # get comments
  {"$group": {
    "_id": "$Comments.UserID", # group comments by user
    "commentCount": {"$sum": 1}
  }},
  {"$setWindowFields": {
    "sortBy": {"commentCount": -1},
    "output": {
      "rank": {"$rank": {}}
    }
  }},
  {"$sort": {"rank": 1}}
]

result4 = pd.DataFrame(list(db.posts.aggregate(query4)))
result4


,_id,commentCount,rank
0,1058,65,1
1,1032,60,2
2,1083,60,2
3,1039,59,4
4,1026,58,5
...,...,...,...
95,1087,34,95
96,1074,33,97
97,1029,33,97
98,1001,32,99


In [24]:
"""
QUERY 5:

daily posting activity using a 7-day rolling avg
"""

query5 = [
  {"$group": {
    "_id": {
      "$dateTrunc": {
        "date": "$Timestamp",
        "unit": "day"
      }
    },
    "dailyPosts": {"$sum": 1}
  }},
  {"$sort": {"_id": 1}},
  {"$setWindowFields": {
    "sortBy": {"_id": 1},
    "output": {
      "rollingAvg7": {
        "$avg": "$dailyPosts",
        "window": {"range": [-6, 0], "unit": "day"}
      }
    }
  }}
]

result5 = pd.DataFrame(db.posts.aggregate(query5))
result5


,_id,dailyPosts,rollingAvg7
0,2024-01-01,2,2.000000
1,2024-01-02,3,2.500000
2,2024-01-03,4,3.000000
3,2024-01-04,5,3.500000
4,2024-01-05,1,3.000000
...,...,...,...
762,2026-02-26,1,2.857143
763,2026-02-27,3,2.428571
764,2026-02-28,3,2.571429
765,2026-03-01,1,2.428571
